In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import glob
import re

import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive

In [2]:
%load_ext autoreload

In [3]:
#Run this to reload the python file
%autoreload 2
from utils import *

# Harmonization

### Importing Data

In [4]:
# local file path
path = './Data/IMN_raw/IMN.xlsx'

**Get sheet names**

In [5]:
file_info = sheet_list(path)

**Reading the info's sheet from the IMN's original file**

In [6]:
# read one sheet into a DataFrame
meta = pd.read_excel(path, sheet_name=file_info.get('info_sheet'))

In [7]:
metadata = info(meta)

### Coordinates transformation

The original coordinates of the IMN are in `Degrees Minutes Seconds` and the desired format is `Decimal Degrees`

In [8]:
decimal_degrees_lat = []

for coord in metadata['Latitud Norte']:
    transf = dms_to_dd(coord)
    decimal_degrees_lat.append(transf)

In [9]:
decimal_degrees_lon = []

for coord in metadata['Longitud Oeste']:
    transf = dms_to_dd(coord)
    transf = -transf
    decimal_degrees_lon.append(transf)

In [10]:
# Add columns for lat and lon in decimal degrees
metadata['lat'] = decimal_degrees_lat
metadata['lon'] = decimal_degrees_lon

In [11]:
metadata.to_csv('./Data/metadata/IMN_stations.csv')

### Sheet format changes

In [12]:
aws_1 = pd.read_excel(path, sheet_name=file_info.get('list_names')[0])

This is the original sheet format of each AWS from IMN, lets to change it to a standard one, where:
- there are no empty rows at the beginning
- columns with corresponding names
- time change from `01:00-23:00` to `00:00-24:00`
- time change from local time to UTC time
    - Costa Rica has one time zone, which is located in the UTC−06:00 zone, 6 hours behind Coordinated Universal Time (UTC)
- save each AWS sheet as an independet file

In [ ]:
%%time

for x in file_info.get('list_names'):
    print(f'Working in {x}')
    try:
        # read one sheet into a DataFrame
        station = pd.read_excel(path, sheet_name=str(x))
        
        # remove empty rows at start of Dataframe
        df = preprocess(station)
    
        # change the format
        df = formating(df)
        
        aws_number = x.replace(' ', '')
    
        # Create DataFrames for pcp data
        df_pcp = pd.DataFrame({'station_number': aws_number, 'date': df['Date'], 'pcp': df['pcp']})
        
        # convert the date column to datetime format
        df_pcp['date'] = pd.to_datetime(df_pcp['date'], infer_datetime_format=True)

        # time change to UTC time
        df_pcp['date'] = df_pcp['date'] + timedelta(hours=6)
    
        # Save the data to separate CSV files
        df_pcp.to_csv(f'./Data/IMN_raw/{aws_number}_IMN.csv', index=False)
    
        print(f'csv created for {x}')
    except Exception as e:
        print(f'Error processing {x}: {str(e)}')
        continue

# Quallity Control

### Importing data

In [13]:
# path file
paths = glob.glob('./Data/IMN_raw/*IMN*csv')

In [15]:
start_date = '2001-01-01 00:00:00'
end_date = '2022-12-31 23:00:00'

In [16]:
percentage = []

for path in paths:
    
    df = pd.read_csv(path)
    
    df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
    
    # checking missing dates
    m_dates = missing_dates(df, 'date', start_date, end_date, 'H')
    
    # create a DataFrame with missing dates and 'NA' in the 'pcp' column
    m_dates = pd.DataFrame({'station_number': path[15:-8], 'date': m_dates, 'pcp': np.nan})
    
    df = pd.concat([df, m_dates], ignore_index=True)
    
    df['date'] = pd.to_datetime(df['date'])  
    
    df = df.sort_values(by='date')
    
    # replace missing values to nan
    df = missing_values(df, 'pcp', -9)
    
    #calculate percentage of missing data in a specific time range
    perc = nan_percentage(df, 'date', 'pcp', start_date, end_date, 'H')
    percentage.append(perc)
    
    if perc < 10:
        df.to_csv(f"./Data/harmonized/{path[15:-8]}_imn.csv")

In [17]:
numbers = []
for path in paths:
    number = path[15:-8]
    numbers.append(number)

In [18]:
tmp = pd.DataFrame()
tmp['station_number'] = numbers
tmp['percentage'] = percentage

In [19]:
# adding the percentage values into the metadata file
meta = pd.read_csv('./Data/metadata/IMN_stations.csv')

In [20]:
meta['Número'] = meta['Número'].astype(str)
merged_df = meta.merge(tmp, left_on='Número', right_on='station_number')

In [21]:
merged_df.to_csv('./Data/metadata/IMN_stations_rev.csv')

In [22]:
merged_df

,Unnamed: 0,Número,Nombre,Latitud Norte,Longitud Oeste,Altitud (m.s.n.m.),Inicio,Fin,lat,lon,station_number,percentage
0,0,69633,"COMANDO LOS CHILES,","11º 01' 54""","84º 42' 42""",40,2000-01-01 00:00:00,2022-12-31 00:00:00,11.031667,-84.711667,69633,8.699440
1,1,69647,FINCA BRASILIA DEL ORO,"10º 58' 59""","85º 20' 50""",380,2002-11-08 00:00:00,2022-12-31 00:00:00,10.983056,-85.347222,69647,12.296723
2,2,69679,UPALA,"10º 52' 51""","85º 04' 21""",60,2000-01-01 00:00:00,2022-12-31 00:00:00,10.880833,-85.072500,69679,4.433727
3,3,69681,LA REBUSCA,"10º 29' 00""","84º 01' 00""",40,2000-01-01 00:00:00,2022-12-31 00:00:00,10.483333,-84.016667,69681,8.064717
4,4,71015,"CANTA GALLO,","10º 29' 48""","83º 40' 28""",20,2000-01-01 00:00:00,2022-12-31 00:00:00,10.496667,-83.674444,71015,9.637523
5,5,72157,"FINCA LA CEIBA,","10º 06' 40""","85º 19' 03""",58,2000-01-01 00:00:00,2022-12-31 00:00:00,10.111111,-85.317500,72157,5.487969
6,6,72159,"PAQUERA,","09º 49' 17""","84º 56' 20""",10,2002-10-27 00:00:00,2022-12-31 00:00:00,9.821389,-84.938889,72159,17.116781
7,7,73123,"ITCR, CARTAGO","09º 51' 08""","83º 54' 31""",1360,2000-01-01 00:00:00,2022-12-31 00:00:00,9.852222,-83.908611,73123,4.155777
8,8,74051,AEROP. LIBERIA OESTE 07,"10º 35' 20,40""","85º 33' 07,70""",89,2000-01-01 00:00:00,2022-12-31 00:00:00,10.589000,-85.552139,74051,1.581622
9,9,74053,"SANTA CRUZ,","10º 17' 07""","85º 35' 30""",40,2000-01-01 00:00:00,2022-12-31 00:00:00,10.285278,-85.591667,74053,8.751815


### Join all files into one

In [21]:
# path file
paths = glob.glob('./Data/harmonized/*imn*csv')

In [22]:
df_comb = pd.DataFrame()

In [23]:
df_comb['date'] = pd.date_range(start=start_date, end=end_date, freq='H')

In [24]:
for path in paths:
    df = pd.read_csv(path)
    
    df_comb[path[18:-8]] = df['pcp']

In [25]:
df_comb.to_csv('./Data/harmonized/unif_IMN.csv', index=False)

# Visualization

In [ ]:
df_comb = pd.read_csv('./Data/harmonized/unif_IMN.csv')

In [ ]:
df = df_comb.copy()
df['date'] = pd.to_datetime(df['date'],infer_datetime_format=True)
df.index = df['date']

In [ ]:
# Create a dropdown widget for stations numbers
station_options = df_comb.columns.difference(['date'])
station_dropdown = widgets.Dropdown(
    options=station_options,
    value=station_options[0],
    description='Station Number:',
    disabled=False
)

In [ ]:
# Define a function to plot the selected station
def plot_station(station_number):
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot(df['date'], df[station_number])
    plt.xlabel('Date')
    plt.ylabel('Precipitation [mm]')
    plt.title(f'Automatic Weather Station: {station_number}')
    plt.xticks(rotation=45)
    plt.show()

# Create an interactive widget
interactive_plot = interactive(plot_station, station_number=station_dropdown)

# Display the interactive widget
interactive_plot